# Retrieval test — does the retriever find the right part of the manual?

**No LLM. No API key. No cost beyond the runtime. A T4 is enough.**

Before any parser work, this answers one question: when a question is worded
freely — not in the manual's own words — does retrieval bring back the section
that holds the answer?

**The test cases come from the MUTCD itself**, not from any test question:
48 provisions picked at random across all nine Parts, each written three ways,
plus 10 questions that need two sections.

| style | what it is |
|---|---|
| `manual_words` | the provision's own text — a plumbing check, should be near 100% |
| `plain_words` | a short question in everyday words, avoiding the manual's terms |
| `scenario` | a few sentences describing a site, with one irrelevant detail |
| `two_section_linked` | needs two sections, one citing the other |
| `two_section_unlinked` | needs two sections with no reference between them |

The file is `evaluation/retrieval_dev/manual_cases_v1.jsonl` — open it and read
the cases. None of the held-out test questions' sections are in it.

**Run every code cell in order: CELL 1 to CELL 8.** CELL 8 compares this run with the earlier run on the old graph.

In [ ]:
# CELL 1 — Drive, repo, packages.
# Same layout as Beyond_RAG_VINE_Colab: data on Drive, repo cloned to local
# disk. MRAG_BASE_DIR must be set BEFORE any mrag import.
import sys, os
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/Beyond_RAG"
REPO_DIR  = "/content/Beyond_RAG_repo"
REPO_URL  = "https://github.com/hannanazad/Beyond_RAG.git"

os.environ["MRAG_ENV"]      = "colab"
os.environ["MRAG_BASE_DIR"] = DRIVE_DIR

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
sys.path.insert(0, REPO_DIR)

# The same installs as the VINE notebook's CELL 1, minus poppler (not needed here).
!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchvision==0.21.0
!pip install -q -r $REPO_DIR/requirements.txt
!pip install -q --no-deps --force-reinstall "transformers>=4.49,<4.55" "huggingface_hub>=0.34,<0.35" "tokenizers>=0.21,<0.22" "torchao>=0.13,<0.14"

assert os.path.exists(f"{REPO_DIR}/evaluation/retrieval_dev/manual_cases_v1.jsonl"), (
    "the test cases are not in the clone -- push evaluation/retrieval_dev/ and "
    "evaluation/retrieval_check.py to GitHub, then re-run this cell")
print("data :", DRIVE_DIR)
print("code :", REPO_DIR)

### The vector store
Qdrant lives on the runtime's local disk (`/content/qdrant_db`) and is lost when
the runtime is recycled. This cell restores it from the snapshot on Drive
(`Beyond_RAG/qdrant_db.tar`) if it is not already there. It never creates an
empty store: if neither exists it stops and says what to run.

In [ ]:
# CELL 2 — make sure the vector store is on local disk.
import shutil, subprocess, time
from pathlib import Path
from mrag.config import CFG

LOCAL = Path(CFG.qdrant_dir)
SNAP  = Path(CFG.base_dir) / "qdrant_db.tar"

def has_store(p: Path) -> bool:
    return (p / "collection" / CFG.coll_chunks).exists()

if has_store(LOCAL):
    print("vector store already on local disk:", LOCAL)
elif SNAP.exists():
    print(f"restoring {SNAP} ({SNAP.stat().st_size / 1e6:.0f} MB) ...")
    t0 = time.time()
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    subprocess.run(["tar", "-xf", str(SNAP), "-C", str(LOCAL.parent)], check=True)
    print(f"  done in {time.time() - t0:.0f}s")
else:
    raise FileNotFoundError(
        f"No vector store at {LOCAL} and no snapshot at {SNAP}.\n"
        "Build it once: run CELL 7 (ingest) of Beyond_RAG_VINE_Colab.ipynb, then\n"
        "  from mrag.colab_setup import snapshot_qdrant_to_drive\n"
        "  snapshot_qdrant_to_drive('Beyond_RAG')\n"
        "and come back here.")

assert has_store(LOCAL), (
    f"the snapshot unpacked, but {LOCAL}/collection/{CFG.coll_chunks} is missing -- "
    "the tar holds a different folder layout. List it with: !tar -tf " + str(SNAP) + " | head")
print("ok:", LOCAL)

### Load retrieval only
Text embedder (BGE-M3), reranker and the retrieval graph. **No image embedder,
no VLM, no API calls.** The store is checked against `chunks.jsonl` so a stale
store is reported, not silently used.

In [ ]:
# CELL 3 — load the retriever (no LLM).
import logging, json, subprocess
logging.basicConfig(level=logging.WARNING)
from mrag.ask import init_pipeline

pipeline = init_pipeline(load_image_embedder=False, load_vlm=False)

n_store = pipeline.store.client.count(CFG.coll_chunks, exact=True).count
n_file  = sum(1 for _ in open(CFG.chunks_jsonl))
ITEMS   = Path(CFG.base_dir) / "vine_data" / "vine_items.jsonl"   # written by Graph_and_Vector_Store.ipynb
n_items = sum(1 for _ in open(ITEMS)) if ITEMS.exists() else 0
print(f"chunks in the vector store : {n_store:,}")
print(f"chunks in chunks.jsonl     : {n_file:,}")
print(f"VINE items (readings, rows): {n_items:,}" + ("" if ITEMS.exists() else "  (none added yet)"))
if n_store != n_file + n_items:
    print(f"WARNING: the store holds {n_store:,}, but chunks.jsonl + VINE items = {n_file + n_items:,}. "
          "Results below describe THAT store; re-run Graph_and_Vector_Store.ipynb or re-ingest.")

mode = getattr(pipeline.text, "_mode", "?")
_, sparse = pipeline.text.encode_both(["minimum mounting height of a sign"])
print(f"text encoder mode          : {mode}")
print(f"sparse weights in a query  : {len(sparse[0])}  (0 = the sparse half is not working; search is dense only)")
print(f"graph                      : {type(pipeline.kg).__name__} from {pipeline.kg.g.graph.get('source', CFG.graph_pickle)}")
print(f"graph size                 : {pipeline.kg.g.number_of_nodes():,} nodes, {pipeline.kg.g.number_of_edges():,} edges")
MANIFEST = Path(CFG.base_dir) / "vine_data" / "text_store_manifest.json"
if MANIFEST.exists():
    print("text store built           :", json.loads(MANIFEST.read_text()))

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
SETTINGS = {k: getattr(CFG, k) for k in (
    "top_k_fused", "top_k_after_graph", "top_k_after_rerank", "top_k_compile_chunks",
    "expansion_source_chunks", "expansion_reserved_slots", "compile_closure_budget")}
print("repo commit                :", COMMIT)
print("retrieval settings         :", SETTINGS)

In [ ]:
# CELL 4 — load the test cases (written from the manual, not from test questions).
sys.path.insert(0, f"{REPO_DIR}/evaluation")
import importlib, retrieval_check as rc
importlib.reload(rc)

cases = rc.load_cases()
from collections import Counter
print(len(cases), "cases:", dict(Counter(c["style"] for c in cases)))
for c in cases[:3]:
    print(f"\n[{c['case_id']}] target {c['target_sections']}\n  {c['text'][:160]}")

In [ ]:
# CELL 5 — run every case through retrieve_for_compile (what the parser reads)
# and retrieve (the RAG baseline). Roughly 5-15 minutes.
import time
t0 = time.time()
results = rc.run_all(pipeline.retriever, cases, with_rag=True)
errors = [r for r in results if "error_compile" in r or "error_rag" in r]
print(f"\ndone in {time.time() - t0:.0f}s | errors: {len(errors)}")
for r in errors[:5]:
    print("  ", r["case_id"], r.get("error_compile") or r.get("error_rag"))

In [ ]:
# CELL 6 — the results.
table = rc.summarize(results)
rc.print_summary(table)

print("\n  scenario cases by Part (target in searched top 5 / in what the parser reads):")
for part, p in rc.by_part(results, "scenario").items():
    print(f"    Part {part}: {p['top5']}/{p['n']} in top 5,  {p['in_kq']}/{p['n']} reach the parser")

In [ ]:
# CELL 7 — every miss, then save everything to Drive.
miss = rc.misses(results)
print(f"{len(miss)} cases where a target section never reached the parser:\n")
text_of = {c["case_id"]: c["text"] for c in cases}
for r in miss:
    print(f"[{r['case_id']}] want {r['target_sections']}  searched top 5: {r.get('searched_top5')}")
    print(f"    {text_of[r['case_id']][:150]}")

out_dir = Path(CFG.base_dir) / "retrieval_dev_results"
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / f"retrieval_test_{time.strftime('%Y%m%d_%H%M%S')}.json"
out.write_text(json.dumps({
    "commit": COMMIT, "settings": SETTINGS, "encoder_mode": mode,
    "graph": type(pipeline.kg).__name__, "graph_source": str(pipeline.kg.g.graph.get("source", CFG.graph_pickle)),
    "store_chunks": n_store, "file_chunks": n_file, "vine_items": n_items,
    "summary": table, "by_part_scenario": rc.by_part(results, "scenario"),
    "results": results}, indent=1, default=str))
print("\nsaved ->", out)

In [ ]:
# CELL 8 — compare with an earlier run: what got better, what got worse.
# BEFORE is the run saved on the OLD graph and old store (7 Oct 2026, 00:49), if it is
# there; otherwise the oldest saved run. AFTER is the run CELL 7 just saved.
runs = sorted(out_dir.glob("retrieval_test_*.json"))
named = out_dir / "retrieval_test_20261007_004919.json"
BEFORE = named if named.exists() else runs[0]
AFTER = out
if BEFORE == AFTER:
    print("only one saved run -- nothing to compare yet")
else:
    before = json.loads(BEFORE.read_text())
    print("before:", BEFORE.name, "| graph", before.get("graph", "KG (old)"), "| store", before.get("store_chunks"), "| VINE items", before.get("vine_items", 0))
    print("after :", AFTER.name, "| graph", type(pipeline.kg).__name__, "| store", n_store, "| VINE items", n_items)
    rc.print_compare(rc.compare(before["results"], results))